# 03 · Treasury bond future: pricing and risk

QuantLib has no bond-future instrument, so we build the standard cash-and-carry model on its bond pricing:

1. Deliverable basket (10Y-note style) priced off a Treasury zero curve.
2. CME **conversion factors** (6% notional coupon).
3. Forward clean price at delivery (repo-financed) → implied futures price per bond → **CTD** and model futures price.
4. Basis analytics: gross basis, carry, net basis, **implied repo**.
5. Risk: **DV01**, key-rate DV01, CTD switch under parallel shifts, and a hedge ratio.

Repo rate = SOFR forward from the curve calibrated in notebook 01. Data is synthetic. The model ignores delivery options other than CTD choice (no wild-card / quality / timing options), so the model price is an upper bound on what a full-option model would give.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # run from notebooks/ so ql_helpers imports
import numpy as np
import QuantLib as ql
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ql_helpers as h
import plot_utils

pio.renderers.default = "plotly_mimetype"
plot_utils.apply()
pd.options.display.float_format = "{:,.4f}".format
print("QuantLib", ql.__version__, "| valuation date", h.set_eval_date())

QuantLib 1.43 | valuation date September 25th, 2026


## 1. Contract and deliverable basket

In [2]:
VAL = h.EVAL_DATE
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
FIRST_DELIVERY = ql.Date(1, 12, 2026)      # first day of delivery month (drives CF maturity rounding)
DELIVERY = ql.Date(31, 12, 2026)           # model delivery date (last delivery day)
CONTRACT_FACE = 100_000                    # USD face per contract

# name, coupon, maturity (synthetic 10Y notes, 6.5-8.0y remaining at first delivery)
BASKET = [
    ("UST 3.875 Aug-33", 0.03875, ql.Date(15, 8, 2033)),
    ("UST 4.125 Nov-33", 0.04125, ql.Date(15, 11, 2033)),
    ("UST 4.000 Feb-34", 0.04000, ql.Date(15, 2, 2034)),
    ("UST 4.375 May-34", 0.04375, ql.Date(15, 5, 2034)),
    ("UST 3.875 Aug-34", 0.03875, ql.Date(15, 8, 2034)),
    ("UST 4.250 Nov-34", 0.04250, ql.Date(15, 11, 2034)),
]

def make_bond(coupon, maturity, curve):
    issue = maturity - ql.Period("10Y")
    sched = ql.Schedule(issue, maturity, ql.Period(ql.Semiannual), CAL, ql.Unadjusted, ql.Unadjusted,
                        ql.DateGeneration.Backward, False)
    bond = ql.FixedRateBond(1, 100.0, sched, [coupon], ql.ActualActual(ql.ActualActual.ISMA))
    bond.setPricingEngine(ql.DiscountingBondEngine(ql.YieldTermStructureHandle(curve)))
    return bond

def conversion_factor(coupon, maturity, first_delivery=FIRST_DELIVERY, yld=0.06):
    """CME conversion factor: price at 6% yield (semi-annual) with maturity rounded down to a quarter."""
    months = (maturity.year() - first_delivery.year()) * 12 + (int(maturity.month()) - int(first_delivery.month()))
    m = 3 * (months // 3)
    n = -(-m // 6)                       # remaining coupons
    t1 = m - 6 * (n - 1)                 # months to first coupon (1..6)
    c = coupon * 100
    df = lambda t: (1 + yld / 2) ** (-t / 6)
    pv = sum(c / 2 * df(t1 + 6 * k) for k in range(n)) + 100 * df(t1 + 6 * (n - 1)) - c / 2 * (6 - t1) / 6
    return round(pv / 100, 4)

assert conversion_factor(0.06, ql.Date(15, 8, 2033)) == 1.0, "6% coupon should give CF = 1"
cf_tbl = pd.DataFrame([{"bond": n, "coupon_%": c * 100, "maturity": m.ISO(), "CF": conversion_factor(c, m)}
                       for n, c, m in BASKET]).set_index("bond")
display(cf_tbl.style.format({"coupon_%": "{:.3f}", "CF": "{:.4f}"}))

,coupon_%,maturity,CF
bond,,,
UST 3.875 Aug-33,3.875,2033-08-15,0.8870
UST 4.125 Nov-33,4.125,2033-11-15,0.8971
UST 4.000 Feb-34,4.000,2034-02-15,0.8870
UST 4.375 May-34,4.375,2034-05-15,0.9055
UST 3.875 Aug-34,3.875,2034-08-15,0.8732
UST 4.250 Nov-34,4.250,2034-11-15,0.8927


## 2. Cash-and-carry model

Per deliverable: forward dirty = dirty·(1 + repo·d/360) − FV(coupons before delivery); forward clean = forward dirty − accrued at delivery; **implied futures price = forward clean / CF**. The CTD minimises this and the model futures price is that minimum.

In [3]:
sofr = h.SofrCurve()
SETTLE = CAL.advance(VAL, 1, ql.Days)
DAYS = DELIVERY - SETTLE
REPO = sofr.curve.forwardRate(SETTLE, DELIVERY, ql.Actual360(), ql.Simple).rate()   # term repo ~ SOFR forward
print("settle %s | delivery %s | %d days | term repo %.4f%%" % (SETTLE.ISO(), DELIVERY.ISO(), DAYS, REPO * 100))

def basket_analytics(curve, repo=REPO):
    rows = []
    for name, coupon, mat in BASKET:
        b = make_bond(coupon, mat, curve)
        clean, dirty = b.cleanPrice(), b.dirtyPrice()
        cpn_fv = sum(cf.amount() * (1 + repo * (DELIVERY - cf.date()) / 360)
                     for cf in b.cashflows() if SETTLE < cf.date() <= DELIVERY)
        fwd_dirty = dirty * (1 + repo * DAYS / 360) - cpn_fv
        acc_del = b.accruedAmount(DELIVERY)
        fwd_clean = fwd_dirty - acc_del
        cf = conversion_factor(coupon, mat)
        rows.append({"bond": name, "CF": cf, "clean": clean, "dirty": dirty, "fwd_clean": fwd_clean,
                     "implied_fut": fwd_clean / cf, "accrued_del": acc_del, "cpn_fv": cpn_fv,
                     "yield_%": b.bondYield(ql.ActualActual(ql.ActualActual.ISMA), ql.Compounded, ql.Semiannual) * 100})
    return pd.DataFrame(rows).set_index("bond")

ust = h.ust_curve()
tbl = basket_analytics(ust)
display(tbl.style.format("{:.4f}").highlight_min(subset="implied_fut", color="#c8f0c8"))
CTD, F_MODEL = tbl["implied_fut"].idxmin(), tbl["implied_fut"].min()
print("CTD: %s | model futures price: %.4f" % (CTD, F_MODEL))

settle 2026-09-28 | delivery 2026-12-31 | 94 days | term repo 3.9808%


,CF,clean,dirty,fwd_clean,implied_fut,accrued_del,cpn_fv,yield_%
bond,,,,,,,,
UST 3.875 Aug-33,0.8870,99.0929,99.5562,99.1379,111.7676,1.4531,0.0000,4.0264
UST 4.125 Nov-33,0.8971,100.5080,102.0325,100.4961,112.0233,0.5242,2.0728,4.0414
UST 4.000 Feb-34,0.8870,99.6292,100.1075,99.6480,112.3428,1.5000,0.0000,4.0578
UST 4.375 May-34,0.9055,101.9985,103.6153,101.9380,112.5765,0.5559,2.1984,4.0668
UST 3.875 Aug-34,0.8732,98.5910,99.0543,98.6308,112.9533,1.4531,0.0000,4.0852
UST 4.250 Nov-34,0.8927,101.0358,102.6064,100.9973,113.1369,0.5401,2.1356,4.0981


CTD: UST 3.875 Aug-33 | model futures price: 111.7676


## 3. Basis analytics vs a market quote

The market price is synthetic: model price minus 3/32 (contract cheap to the model). Per bond:

* gross basis = clean − CF × F
* carry = clean − forward clean (coupon income − financing to delivery)
* net basis = gross basis − carry, i.e. forward clean − CF × F (≈ delivery-option value; 0 for the CTD when F is the model price)
* implied repo = financing rate that makes buy-bond / sell-future / deliver break even (coupon reinvestment held at the term repo)

In [4]:
F_MKT = round(F_MODEL - 3 / 32, 4)

def basis_table(tbl, fut):
    out = pd.DataFrame(index=tbl.index)
    out["CF"] = tbl["CF"]
    out["clean"] = tbl["clean"]
    out["gross_basis"] = tbl["clean"] - tbl["CF"] * fut
    out["carry"] = tbl["clean"] - tbl["fwd_clean"]
    out["net_basis"] = out["gross_basis"] - out["carry"]
    invoice = tbl["CF"] * fut + tbl["accrued_del"]
    out["implied_repo_%"] = ((invoice + tbl["cpn_fv"]) / tbl["dirty"] - 1) * 360 / DAYS * 100
    return out

print("Market futures %.4f | model %.4f | term repo %.4f%%" % (F_MKT, F_MODEL, REPO * 100))
bt = basis_table(tbl, F_MKT)
display(bt.style.format("{:.4f}").highlight_min(subset="net_basis", color="#c8f0c8").highlight_max(subset="implied_repo_%", color="#c8f0c8"))

bt_fair = basis_table(tbl, F_MODEL)
print("At the model price the CTD net basis is %.6f and its implied repo %.4f%% (= term repo %.4f%%)"
      % (bt_fair.loc[CTD, "net_basis"], bt_fair.loc[CTD, "implied_repo_%"], REPO * 100))
assert abs(bt_fair.loc[CTD, "net_basis"]) < 1e-9

Market futures 111.6739 | model 111.7676 | term repo 3.9808%


,CF,clean,gross_basis,carry,net_basis,implied_repo_%
bond,,,,,,
UST 3.875 Aug-33,0.8870,99.0929,0.0381,-0.0450,0.0831,3.6610
UST 4.125 Nov-33,0.8971,100.5080,0.3254,0.0119,0.3135,2.8043
UST 4.000 Feb-34,0.8870,99.6292,0.5745,-0.0188,0.5933,1.7111
UST 4.375 May-34,0.9055,101.9985,0.8778,0.0605,0.8173,0.9599
UST 3.875 Aug-34,0.8732,98.5910,1.0774,-0.0398,1.1172,-0.3385
UST 4.250 Nov-34,0.8927,101.0358,1.3445,0.0385,1.3060,-0.8940


At the model price the CTD net basis is 0.000000 and its implied repo 3.9808% (= term repo 3.9808%)


## 4. Risk: DV01 and key-rate DV01

Bump the Treasury zero curve and re-run the whole model (including CTD selection) to get futures DV01. Repo is held constant. Contract value = 1,000 × price ($100k face / 100).

In [5]:
def model_future(**bump):
    t = basket_analytics(h.ust_curve(**bump))
    return t["implied_fut"].min(), t["implied_fut"].idxmin()

FUT_PT = CONTRACT_FACE / 100   # USD per price point per contract
dv01_fut = (model_future(parallel_bp=-0.5)[0] - model_future(parallel_bp=0.5)[0]) * FUT_PT   # USD/contract per +1bp

# cash CTD DV01 per $100k face and its CF-adjusted equivalent
ctd_coupon, ctd_mat = next((c, m) for n, c, m in BASKET if n == CTD)
def bond_value(face=CONTRACT_FACE, **bump):
    return make_bond(ctd_coupon, ctd_mat, h.ust_curve(**bump)).NPV() * face / 100
dv01_ctd = (bond_value(parallel_bp=-0.5) - bond_value(parallel_bp=0.5))
cf_ctd = tbl.loc[CTD, "CF"]
print("Futures DV01           : $%.2f per contract per bp" % dv01_fut)
print("CTD bond DV01 ($100k)   : $%.2f  -> / CF = $%.2f  (CF-adjusted CTD DV01 approximates futures DV01)" % (dv01_ctd, dv01_ctd / cf_ctd))

Futures DV01           : $68.89 per contract per bp
CTD bond DV01 ($100k)   : $60.54  -> / CF = $68.25  (CF-adjusted CTD DV01 approximates futures DV01)


In [6]:
# key-rate DV01 of the future vs the CTD (per contract / per $100k face)
rows = {}
for tenor in h.UST_ZERO_RATES:
    up = model_future(bumps_bp={tenor: +0.5})[0]; dn = model_future(bumps_bp={tenor: -0.5})[0]
    b_up = bond_value(bumps_bp={tenor: +0.5});    b_dn = bond_value(bumps_bp={tenor: -0.5})
    rows[tenor] = {"future_KRD": (dn - up) * CONTRACT_FACE / 100, "CTD_KRD / CF": (b_dn - b_up) / cf_ctd}
krd = pd.DataFrame(rows).T
display(krd.style.format("{:.3f}"))
print("sum future KRD = %.2f vs parallel %.2f" % (krd["future_KRD"].sum(), dv01_fut))

fig = go.Figure([go.Bar(x=krd.index, y=krd[c], name=c) for c in krd.columns])
fig.update_layout(barmode="group", title="Key-rate DV01: future vs CF-adjusted CTD (USD per contract per bp)", height=400)
fig.show()

,future_KRD,CTD_KRD / CF
3M,-0.058,0.035
6M,0.092,0.091
1Y,0.370,0.366
2Y,0.818,0.809
3Y,1.936,1.915
5Y,6.963,6.889
7Y,58.771,58.148
10Y,0.000,0.000
20Y,0.000,0.000
30Y,0.000,0.000


sum future KRD = 68.89 vs parallel 68.89


## 5. Scenarios: CTD switch and negative convexity

Shift the Treasury curve in parallel and re-run. As yields move, the CTD can switch, which makes the future's payoff kinked/less convex than any single bond.

In [7]:
rows, per_bond = [], {}
shifts = list(range(-200, 201, 25))
for sh in shifts:
    t = basket_analytics(h.ust_curve(parallel_bp=sh))
    rows.append({"shift_bp": sh, "F_model": t["implied_fut"].min(), "CTD": t["implied_fut"].idxmin()})
    per_bond[sh] = t["implied_fut"]
sc = pd.DataFrame(rows).set_index("shift_bp")
display(sc.style.format({"F_model": "{:.4f}"}))
print("CTD changes across scenarios:", sc["CTD"].nunique(), "distinct ->", list(sc["CTD"].unique()))

pb = pd.DataFrame(per_bond).T
fig = go.Figure()
for b in pb.columns:
    fig.add_scatter(x=pb.index, y=pb[b], name=b, opacity=0.6)
fig.add_scatter(x=sc.index, y=sc["F_model"], name="futures = min (CTD)", line=dict(color="black", width=4))
fig.update_layout(title="Implied futures price per deliverable vs parallel shift", xaxis_title="bp", yaxis_title="price", height=450)
fig.show()

,F_model,CTD
shift_bp,,
-200,126.4948,UST 3.875 Aug-33
-175,124.5460,UST 3.875 Aug-33
-150,122.6290,UST 3.875 Aug-33
-125,120.7434,UST 3.875 Aug-33
-100,118.8887,UST 3.875 Aug-33
-75,117.0642,UST 3.875 Aug-33
-50,115.2695,UST 3.875 Aug-33
-25,113.5042,UST 3.875 Aug-33
0,111.7676,UST 3.875 Aug-33


CTD changes across scenarios: 2 distinct -> ['UST 3.875 Aug-33', 'UST 4.250 Nov-34']


## 6. Hedge ratio

Hedge a long $100mm position in the CTD with futures. Compare the naive DV01 hedge with the resulting P&L under shifts and a curve twist.

In [8]:
FACE = 100e6
dv01_pos = (bond_value(FACE, parallel_bp=-0.5) - bond_value(FACE, parallel_bp=0.5))
n_fut = dv01_pos / dv01_fut
print("Position DV01 $%s | futures DV01 $%.2f | short %.0f contracts" % (f"{dv01_pos:,.0f}", dv01_fut, n_fut))

f0, p0 = model_future()[0], bond_value(FACE)
scen = {"+50bp parallel": dict(parallel_bp=50), "-50bp parallel": dict(parallel_bp=-50),
        "+100bp parallel": dict(parallel_bp=100), "-100bp parallel": dict(parallel_bp=-100),
        "7Y +25bp (twist)": dict(bumps_bp={"7Y": 25}), "5Y +25bp (twist)": dict(bumps_bp={"5Y": 25})}
rows = []
for name, kw in scen.items():
    bond_pnl = bond_value(FACE, **kw) - p0
    fut_pnl = -n_fut * (model_future(**kw)[0] - f0) * CONTRACT_FACE / 100
    rows.append({"scenario": name, "bond_pnl": bond_pnl, "short_futures_pnl": fut_pnl, "net": bond_pnl + fut_pnl})
hedge = pd.DataFrame(rows).set_index("scenario")
display(hedge.style.format("{:,.0f}"))

Position DV01 $60,541 | futures DV01 $68.89 | short 879 contracts


,bond_pnl,short_futures_pnl,net
scenario,,,
+50bp parallel,"-2,977,756","2,977,812",56
-50bp parallel,"3,077,467","-3,077,409",58
+100bp parallel,"-5,859,105","5,859,323",218
-100bp parallel,"6,258,062","-6,257,823",239
7Y +25bp (twist),"-1,279,116","1,280,846","1,731"
5Y +25bp (twist),"-152,402","152,608",206
